# Challenge 2 — Data Science 1

**Student: Fabiana Rotella Campanari**

Run each cell and **write your answer** in the indicated field. You **do not** need to write code — the goal is to interpret and compare the results. The `.csv` files are in this folder.

> **Rule of this challenge:** every statement that mentions a value must **include the value**, copied from the output of its cell. Numbers change from notebook to notebook.

In [7]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import (train_test_split, cross_val_score, KFold,
                                     StratifiedKFold, GroupKFold, TimeSeriesSplit)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def modelo():
    """The model used in simple base questions."""
    return Pipeline([("esc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000))])

def floresta():
    """The model used in panel and series questions."""
    return RandomForestClassifier(n_estimators=60, random_state=0)

print("environment ready")

ambiente pronto


## Exercise 1 — A single split is like flipping a coin

The `q1_amostra.csv` dataset has 300 customers, 8 variables, and a binary target. The code trains the SAME model ten times, changing only the **seed of the split** that separates train and test (70/30).

**(a)** Give the **lowest** and **highest** scores among the ten, and the **range** (highest − lowest).

**(b)** If you had only run seed 0 and stopped there, what number would you have announced? Why is announcing that number a problem?

**(c)** Each of the ten measurements is **honest** — the test set was never used in training. So is the single split problem one of **bias** or **instability**? Justify.

In [16]:
df = pd.read_csv("q1_amostra.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values

notas = []
for s in range(10):
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.3, random_state=s, stratify=y)
    nota = modelo().fit(Xtr, ytr).score(Xte, yte)
    notas.append(nota)
    print(f"seed {s}: {nota:.4f}")

notas = np.array(notas)
print(f"\nlowest {notas.min():.4f} | highest {notas.max():.4f}"
      f" | range {notas.max() - notas.min():.4f}")

seed 0: 0.6667
seed 1: 0.7111
seed 2: 0.7111
seed 3: 0.6000
seed 4: 0.6444
seed 5: 0.7333
seed 6: 0.7222
seed 7: 0.7111
seed 8: 0.7000
seed 9: 0.7111

lowest 0.6000 | highest 0.7333 | range 0.1333


**Answer:**

A- Lowest: 0.6000, Highest: 0.7333, Range: 0.1333.

B- I would announce **0.6667**. Problem: too variable, can be misleading.

C- The score seems to show instability, it changes a lot depending on the split.

## Exercise 2 — The same group on both sides

The `q5_painel.csv` dataset has **40 companies tracked for 16 quarters** — 640 rows, but only 40 distinct companies. The target indicates whether the company outperformed the sector in that quarter. The code compares random shuffling with `GroupKFold`, which keeps each entire company on only one side.

**(a)** Give the **two scores** and the **drop** between them.

**(b)** Explain the mechanism: what does random sampling do with the 16 rows of the same company, and what does the model start to do because of it?

**(c)** Which of the two scores would you report for "this model will be used on companies that do not yet exist in the dataset"? And the other score — what question does it answer?

In [17]:
df = pd.read_csv("q5_painel.csv")
cols = ["porte", "margem", "crescimento", "endividamento", "liquidez"]
X = df[cols].values
y = df["superou_setor"].values
g = df["id_empresa"].values
print("rows:", len(df), "| distinct companies:", df.id_empresa.nunique(),
      "| quarters:", df.periodo.nunique())

aleat = cross_val_score(floresta(), X, y,
                        cv=KFold(5, shuffle=True, random_state=0)).mean()
grupo = cross_val_score(floresta(), X, y, cv=GroupKFold(5), groups=g).mean()
print(f"\nrandom shuffle : {aleat:.4f}")
print(f"GroupKFold        : {grupo:.4f}")
print(f"drop             : {aleat - grupo:.4f}")

rows: 640 | distinct companies: 40 | quarters: 16

random shuffle : 0.7609
GroupKFold        : 0.6000
drop             : 0.1609


**Answer:**

A- Random split: 0.7609, GroupKFold: **0.6000**, Drop: **0.1609**.

B- It mixes data from the same company (train/test), the model 'cheats', inflating performance.

C- For new companies: **GroupKFold (0.6000)**. The other (0.7609) is for already known companies.

## Exercise 3 — The future within the training data

The `q6_serie.csv` dataset is a time series of 600 quarters **in order**. The code compares random shuffling with `TimeSeriesSplit`, which always trains on the past and tests on the future. The last cell shows how the relationship between indicators and the target behaves along the series.

**(a)** Give the **two scores** and the **drop** between them.

**(b)** Why does shuffling the rows put the future into the training data — and, looking at the last cell, why does this weigh so much in this series?

**(c)** The dataset from the company question also has a period column. Did using `GroupKFold` there also solve the time problem? Justify.

In [18]:
df = pd.read_csv("q6_serie.csv")          # already in quarter order
cols = ["trimestre", "ind_a", "ind_b", "ind_c", "ind_d"]
X = df[cols].values
y = df["alvo"].values
print("quarters:", df.trimestre.min(), "to", df.trimestre.max())

aleat = cross_val_score(floresta(), X, y,
                        cv=KFold(5, shuffle=True, random_state=0)).mean()
tempo = cross_val_score(floresta(), X, y, cv=TimeSeriesSplit(5)).mean()
print(f"\nrandom shuffle : {aleat:.4f}")
print(f"TimeSeriesSplit   : {tempo:.4f}")
print(f"drop             : {aleat - tempo:.4f}")

quarters: 0 to 599

random shuffle : 0.8733
TimeSeriesSplit   : 0.6980
drop             : 0.1753


In [19]:
# The relationship between ind_a and the target, in the first and second halves of the series:
meio = len(df) // 2
for rot, parte in [("1st half", df.iloc[:meio]), ("2nd half", df.iloc[meio:])]:
    r = np.corrcoef(parte["ind_a"], parte["alvo"])[0, 1]
    print(f"{rot}: correlation between ind_a and target = {r:+.3f}")

1st half: correlation between ind_a and target = +0.591
2nd half: correlation between ind_a and target = -0.653


**Answer:**

**(a)** Random split: **0.8733**, TimeSeriesSplit: **0.6980**, Drop: **0.1753**.

**(b)** It mixes future data into the training set. The relationship between ind_a/target changes significantly (from +0.591 to -0.653), 'peeking' helps too much.

**(c)** No. `GroupKFold` separates companies, but not the order of time. Temporal problems persist.

## Exercise 4 — When the class is too rare to shuffle

The `q4_raros.csv` dataset has 200 rows and very few positives. The code shows how many positives fall into each fold with common `KFold` and with `StratifiedKFold`, and the F1 score for each fold in both cases.

**(a)** How many positives does the dataset have, and how many fell into each fold with the common `KFold`?

**(b)** What happens to the F1 score in the fold that ended up without any positives — and why?

**(c)** With `StratifiedKFold`, what changed in the distribution of positives and in the F1 score?

**(d)** What is the **largest k** that makes sense in this dataset? Give the number and justify.

In [20]:
df = pd.read_csv("q4_raros.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("rows:", len(y), "| positives:", int(y.sum()),
      f"({100*y.mean():.1f}%)")

for nome, cv in [("KFold common      ", KFold(5, shuffle=True, random_state=0)),
                 ("StratifiedKFold  ", StratifiedKFold(5, shuffle=True, random_state=0))]:
    partes = list(cv.split(X, y))
    pos = [int(y[te].sum()) for _, te in partes]
    f1 = cross_val_score(modelo(), X, y, cv=partes, scoring="f1")
    print(f"\n{nome} positives per fold: {pos}")
    print(f"{' '*19}F1 per fold       : {np.round(f1, 3)}")
    print(f"{' '*19}Average F1           : {f1.mean():.4f}")

rows: 200 | positives: 9 (4.5%)

KFold common       positives per fold: [2, 1, 2, 4, 0]
                   F1 per fold       : [0.    0.5   0.667 0.4   0.   ]
                   Average F1           : 0.3133

StratifiedKFold   positives per fold: [1, 2, 2, 2, 2]
                   F1 per fold       : [0.667 0.667 0.    0.667 0.667]
                   Average F1           : 0.5333


**Answer:**

A- The dataset has 200 rows, with 9 positive rows, which is 4.5% of the total. With the normal KFold, the positives that fell into each part were: {2, 1, 2, 4, 0}.

--

B- In the part that had no positives (0) in [2, 1, 2, 4, 0], the F1 score was **0.000**. This happens because F1 needs to find true positives. If there are no positives to find in the test set, it cannot calculate and returns zero.

--

C- With StratifiedKFold, the positives were better distributed: [1, 2, 2, 2, 2]. It ensures that each part has roughly the same proportion of positives. Therefore, the F1 of each part was more 'normal': [0.667, 0.667, 0.000, 0.667, 0.667] (I will correct this to match the output: [0.667, 0.667, 0.000, 0.667, 0.667]), and the average F1 was 0.5333 (much better than the previous 0.3133, which had two parts with F1=0).

--

D- The maximum sensible k is **9**. Because the dataset has 9 positives. For each part of the test to have at least one positive, the number of parts k cannot be greater than the total number of positives. If it were more than 9, some parts would be without positives, and the F1 calculation would cause problems.

## Exercise 5 — The preprocessing that leaks

The `q3_muitas_colunas.csv` dataset has **120 rows and 180 columns**. The code does the same thing in two ways: first, it selects the 15 best columns by looking at the entire dataset and **only then** validates; second, the column selection enters the `Pipeline` and is redone within each fold.

**(a)** Give the **two scores** and the **difference** between them.

**(b)** The first score is a lie. What is the mechanism — what exactly did `SelectKBest` see that it shouldn't have?

**(c)** Run the last cell, which reveals how the target of this dataset was constructed. In light of this, what is the **real** possible performance here — and what does the `Pipeline` do differently within cross-validation?

In [21]:
df = pd.read_csv("q3_muitas_colunas.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values
print("format:", X.shape, "  <- 120 rows, 180 columns")

cv = StratifiedKFold(5, shuffle=True, random_state=0)

# (1) WRONG: selects columns by looking at the y of ALL rows
sel = SelectKBest(f_classif, k=15).fit(X, y)
errado = cross_val_score(LogisticRegression(max_iter=1000),
                         sel.transform(X), y, cv=cv).mean()

# (2) CORRECT: selection enters the Pipeline and is redone in each fold
certo = cross_val_score(
    Pipeline([("sel", SelectKBest(f_classif, k=15)),
              ("clf", LogisticRegression(max_iter=1000))]), X, y, cv=cv).mean()

print(f"\n(1) selection BEFORE validation : {errado:.4f}")
print(f"(2) selection INSIDE the Pipeline : {certo:.4f}")
print(f"    difference                  : {errado - certo:.4f}")

format: (120, 180)   <- 120 rows, 180 columns

(1) selection BEFORE validation : 0.7750
(2) selection INSIDE the Pipeline : 0.5583
    difference                  : 0.2167


In [22]:
# How the target of this dataset was constructed:
print("proportion of 1s:", round(y.mean(), 3))
print("average correlation |r| between columns and target:",
      round(float(np.abs(np.corrcoef(X.T, y)[-1, :-1]).mean()), 4))
print("\nThe target was RANDOMLY DRAWN (coin toss), with no relationship")
print("with the 180 columns. There is no pattern to learn.")

proportion of 1s: 0.433
average correlation |r| between columns and target: 0.077

The target was RANDOMLY DRAWN (coin toss), with no relationship
with the 180 columns. There is no pattern to learn.


**Answer:**

The final test set is isolated from cross-validation and is touched only once, for the final measurement of the model adjusted on all development data.

Same dataset as the single split question. Now, instead of a single split, the shuffle: each piece is test once and train in the others. The code runs with **k = 5** and with **k = 10.**

**A.** Give the average score and the **standard deviation** for each of the two (k = 5 and k = 10).

**B.** Compare the standard deviation of k-fold with the **range of the hold-out** from the other question, citing both numbers. What does the comparison show?

**C.** What exactly does the standard deviation returned by cross-validation measure? And what changes, in practice, when going from k = 5 to k = 10?

In [23]:
df = pd.read_csv("q1_amostra.csv")
X = df.drop(columns="alvo").values
y = df["alvo"].values

for k in [5, 10]:
    cv = StratifiedKFold(k, shuffle=True, random_state=0)
    notas = cross_val_score(modelo(), X, y, cv=cv)
    print(f"k={k:2d} | scores: {np.round(notas, 3)}")
    print(f"       mean {notas.mean():.4f} +/- {notas.std():.4f}\n")

k= 5 | scores: [0.667 0.717 0.667 0.767 0.667]
       mean 0.6967 +/- 0.0400

k=10 | scores: [0.733 0.633 0.767 0.667 0.733 0.533 0.8   0.767 0.633 0.733]
       mean 0.7000 +/- 0.0775



**Answer:**

A- k=5: mean **0.6967** +/- **0.0400**. k=10: mean **0.7000** +/- **0.0775**.

B- K-fold standard deviation (0.0400 or 0.0775) < hold-out range (0.1333). K-fold is more stable.

C- It measures the **variability** of performance. From k=5 to k=10: training uses more data, the estimate is closer to the true value and with less variation. The computational cost increases.